# Mentor Scoring System
We want to give **every mentor a score** (between 0 and 1) based on:

**P** Student Progress - Did students finish their milestones?

**R** Responsiveness - Did the mentor reply fast?

**E** Engagement - Did they hold meetings, do code reviews?

**F** Feedback - Did students rate the mentor highly?

My Assumption formula:

M = (0.35 × P) + (0.20 × R) + (0.25 × E) + (0.20 × F)

P = 0.35 - This is most important because it shows if students actually learned something and completed their work.

E = 0.25 - This is second because meetings and reviews help us understand how much effort was put in.

R = 0.20 - This is also important because if mentors are slow, students can get frustrated.

F = 0.20 - This has a lower weight because feedback is based on opinion and can be different for everyone.

In [23]:
%pip install numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd   
import numpy as np 

In [25]:
mentor_table = pd.read_csv("../datasets/mentors.csv")
mentor_table

,MentorID,Name,Domain,Projects
0,M001,Alice Sharma,ML,"P001,P002"
1,M002,Bob Mehta,Web,P003
2,M003,Carol Singh,CP,"P004,P005"
3,M004,Dev Patel,Systems,P006
4,M005,Eva Nair,ML,P007


In [26]:
student_table = pd.read_csv("../datasets/students.csv")
student_table

,StudentID,Name,ProjectID,MilestonesCompleted,TotalMilestones,Feedback
0,S001,Ravi Kumar,P001,4,5,4
1,S002,Sneha Das,P001,5,5,5
2,S003,Arjun Rao,P002,3,5,3
3,S004,Priya Iyer,P003,5,5,5
4,S005,Karan Joshi,P003,4,5,4
5,S006,Meera Nair,P003,2,5,2
6,S007,Anil Gupta,P004,5,5,5
7,S008,Divya Sen,P004,4,5,4
8,S009,Rahul Tiwari,P005,1,5,1
9,S010,Pooja Verma,P006,3,5,3


In [27]:
interaction_table = pd.read_csv("../datasets/interactions.csv")
interaction_table

,MentorID,StudentID,Meetings,CodeReviews,Messages,AvgResponseTime
0,M001,S001,5,4,30,2.0
1,M001,S002,6,5,40,1.5
2,M001,S003,3,2,20,3.0
3,M002,S004,8,6,60,0.5
4,M002,S005,7,5,50,1.0
5,M002,S006,2,1,10,8.0
6,M003,S007,6,5,45,2.0
7,M003,S008,5,4,35,2.5
8,M003,S009,1,0,5,20.0
9,M004,S010,4,3,25,5.0


- I want to check which student belongs to which mentor so I am combining student table with intrection table

In [28]:
student_cols = ["StudentID", "MilestonesCompleted", "TotalMilestones", "Feedback"]

In [29]:
combined_data = pd.merge(
    interaction_table,                        
    student_table[student_cols],       
    on="StudentID",                           
    how="left"                                 
)

In [30]:
combined_data

,MentorID,StudentID,Meetings,CodeReviews,Messages,AvgResponseTime,MilestonesCompleted,TotalMilestones,Feedback
0,M001,S001,5,4,30,2.0,4,5,4
1,M001,S002,6,5,40,1.5,5,5,5
2,M001,S003,3,2,20,3.0,3,5,3
3,M002,S004,8,6,60,0.5,5,5,5
4,M002,S005,7,5,50,1.0,4,5,4
5,M002,S006,2,1,10,8.0,2,5,2
6,M003,S007,6,5,45,2.0,5,5,5
7,M003,S008,5,4,35,2.5,4,5,4
8,M003,S009,1,0,5,20.0,1,5,1
9,M004,S010,4,3,25,5.0,3,5,3


Now, I am going to measure P Score, means did the mentor's students actually finish their work?

**simple formula**

P = milestones completed / total milestones

If student finished 4 out of 5 then P = 0.8

I am assuming, if student MilestonesCompleted value is 3 it means he already completed 1,2,3 milestone.

so, Completed milestones = 1, 2, 3

Their weights = 1/5, 2/5, 3/5

Sum of weights = (1+2+3)/5 = 6/5 = 1.2

Max possible = (1+2+3+4+5)/5 = 15/5 = 3.0

Weighted P  = 1.2 / 3.0 = 0.4

** I am using this shortcut formula to calculate weight **

n = n×(n+1)/2

I will get complete weight between 0-1

In [31]:
def calculate_weighted_progress(done, total):
    completed_weight = done * (done + 1) / 2 / total
    max_weight = total * (total + 1) / 2 / total
    return completed_weight / max_weight


def get_weighted_progress(row):
    weight = calculate_weighted_progress(row["MilestonesCompleted"], row["TotalMilestones"])
    return round(weight,2)

In [32]:
first_row = combined_data.iloc[0]
get_weighted_progress(first_row)

np.float64(0.67)

In [33]:
combined_data["student_progress"] = combined_data.apply(
    get_weighted_progress,
    axis=1
)

In [34]:
combined_data

,MentorID,StudentID,Meetings,CodeReviews,Messages,AvgResponseTime,MilestonesCompleted,TotalMilestones,Feedback,student_progress
0,M001,S001,5,4,30,2.0,4,5,4,0.67
1,M001,S002,6,5,40,1.5,5,5,5,1.00
2,M001,S003,3,2,20,3.0,3,5,3,0.40
3,M002,S004,8,6,60,0.5,5,5,5,1.00
4,M002,S005,7,5,50,1.0,4,5,4,0.67
5,M002,S006,2,1,10,8.0,2,5,2,0.20
6,M003,S007,6,5,45,2.0,5,5,5,1.00
7,M003,S008,5,4,35,2.5,4,5,4,0.67
8,M003,S009,1,0,5,20.0,1,5,1,0.07
9,M004,S010,4,3,25,5.0,3,5,3,0.40


Now Average progress per mentor

In [35]:
progress_score = combined_data.groupby("MentorID")["student_progress"].mean()

In [36]:
progress_score

MentorID
M001    0.690000
M002    0.623333
M003    0.580000
M004    0.400000
M005    0.890000
Name: student_progress, dtype: float64

In [37]:
progress_score.name = "P"

In [38]:
progress_score

MentorID
M001    0.690000
M002    0.623333
M003    0.580000
M004    0.400000
M005    0.890000
Name: P, dtype: float64

##### R Score: - how fast does the mentor reply?

I have to make score between 0 to 1 so I am using Exponential formula

R = e^(−0.1 x avg_response_time_in_hours)

λ (lambda) = 0.1 - This is just a number that controls how fast the score goes down.
If λ is bigger, the score will drop faster.
If λ is smaller, the score will drop slowly.

We use this because we want to reduce the score when time increases

Here, λ = 0.1 means the score becomes about half when time reaches around 10 hours.



if mentor give instant reply means 0 hours so score will be
**e^0 = 1.0**

10 hours e^(−1.0) = **0.37**

48 hours e^(−4.8) = **0.008** this is nearly 0

- now Average response time per mentor

In [39]:
avg_response_time = combined_data.groupby("MentorID")["AvgResponseTime"].mean()

In [40]:
avg_response_time

MentorID
M001    2.166667
M002    3.166667
M003    8.166667
M004    5.000000
M005    1.166667
Name: AvgResponseTime, dtype: float64

In [41]:
lambda_value = 0.1  

responsiveness_score = np.exp(-lambda_value * avg_response_time).round(4)

In [42]:
responsiveness_score

MentorID
M001    0.8052
M002    0.7286
M003    0.4419
M004    0.6065
M005    0.8899
Name: AvgResponseTime, dtype: float64

In [43]:
responsiveness_score.name = "R"

In [44]:
responsiveness_score

MentorID
M001    0.8052
M002    0.7286
M003    0.4419
M004    0.6065
M005    0.8899
Name: R, dtype: float64

####  Score E: Engagement

how much effort does the mentor put into interacting with students?

I am assuming weights like this

meeting_weight = 0.4 - needs real time commitment

code_review_weight = 0.4 - needs technical effort

message_weight = 0.2 - easier, so lower weight

In [45]:
meeting_weight = 0.4
code_review_weight = 0.4
message_weight = 0.2

In [46]:
engagment = (meeting_weight * combined_data["Meetings"] + 
             code_review_weight * combined_data["CodeReviews"] +
             message_weight * combined_data["Messages"]
)

In [47]:
engagment

0      9.6
1     12.4
2      6.0
3     17.6
4     14.8
5      3.2
6     13.4
7     10.6
8      1.4
9      7.8
10    16.2
11    13.4
12    15.2
dtype: float64

In [48]:
combined_data["engagment"] = engagment.round(3)

In [49]:
combined_data

,MentorID,StudentID,Meetings,CodeReviews,Messages,AvgResponseTime,MilestonesCompleted,TotalMilestones,Feedback,student_progress,engagment
0,M001,S001,5,4,30,2.0,4,5,4,0.67,9.6
1,M001,S002,6,5,40,1.5,5,5,5,1.00,12.4
2,M001,S003,3,2,20,3.0,3,5,3,0.40,6.0
3,M002,S004,8,6,60,0.5,5,5,5,1.00,17.6
4,M002,S005,7,5,50,1.0,4,5,4,0.67,14.8
5,M002,S006,2,1,10,8.0,2,5,2,0.20,3.2
6,M003,S007,6,5,45,2.0,5,5,5,1.00,13.4
7,M003,S008,5,4,35,2.5,4,5,4,0.67,10.6
8,M003,S009,1,0,5,20.0,1,5,1,0.07,1.4
9,M004,S010,4,3,25,5.0,3,5,3,0.40,7.8


- engagement Average per mentor

In [50]:
avg_engagement = combined_data.groupby("MentorID")["engagment"].mean().round(3)

In [51]:
avg_engagement

MentorID
M001     9.333
M002    11.867
M003     8.467
M004     7.800
M005    14.933
Name: engagment, dtype: float64

In [52]:
lowest_engagement = avg_engagement.min()
highest_engagement = avg_engagement.max()

In [53]:
lowest_engagement

np.float64(7.8)

In [54]:
highest_engagement

np.float64(14.933)

In [55]:
engagement_score = (avg_engagement - lowest_engagement) / (highest_engagement - lowest_engagement)

In [56]:
engagement_score = engagement_score.round(2)

In [57]:
engagement_score

MentorID
M001    0.21
M002    0.57
M003    0.09
M004    0.00
M005    1.00
Name: engagment, dtype: float64

In [58]:
engagement_score.name = "E"

In [59]:
engagement_score

MentorID
M001    0.21
M002    0.57
M003    0.09
M004    0.00
M005    1.00
Name: E, dtype: float64

####  Score F: Feedback
Students rated mentors 1 to 5. I will convert this into a 0-1 score.

##### Problem 1: Biased ratings
One angry student might give 1/5 even if the mentor is great.

**Fix - Trimmed Mean**: If mentor has 4+ students, remove the single highest and lowest rating before averaging.
This is exactly how **Olympic judging** works

##### Problem 2: Small number of students
If a mentor has only 1 student who gave 5/5, that's not enough to trust fully.

- after search I got this formula

**Fix - Bayesian Average**:

adjusted_rating = (n × mean + 3 × global_mean) / (n + 3)
- n = number of students
- global_mean = average rating across ALL mentors
- 3 = I assume that there are 3 extra "neutral" students (our confidence level)

I am assuming 
weight **1.0** for normal rating and 
**0.1** when rating is very far from others, and I got to know that it is called outlier.

and this can be handle by **IQR method**

IQR method:
- Q1 = value at the 25th percentile (bottom quarter)

- Q3 = value at the 75th percentile (top quarter)

- IQR = Q3 - Q1

- Anything below Q1 - 1.5*IQR or above Q3 + 1.5*IQR = outlier
    

In [60]:
def get_weights_for_ratings(all_ratings):
    if len(all_ratings) < 3:
        return [1.0] * len(all_ratings)
    
    ratings_array = np.array(all_ratings)
    Q1  = np.percentile(ratings_array, 25)   
    Q3  = np.percentile(ratings_array, 75)   
    IQR = Q3 - Q1                           

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    weights = []
    for rating in all_ratings:
        if lower_limit <= rating <= upper_limit:
            weights.append(1.0)  
        else:
            weights.append(0.1)  
    
    return weights

    

In [61]:
test = [4, 5, 4, 5, 1]   # 1 is suspicious
test

[4, 5, 4, 5, 1]

In [62]:

get_weights_for_ratings(test)


[1.0, 1.0, 1.0, 1.0, 0.1]

Collect ratings for each mentor

In [63]:
mentor_ratings = combined_data.groupby("MentorID")["Feedback"].apply(list)

In [64]:
mentor_ratings

MentorID
M001    [4, 5, 3]
M002    [5, 4, 2]
M003    [5, 4, 1]
M004          [3]
M005    [5, 4, 1]
Name: Feedback, dtype: object

In [65]:
feedback_scores = {}   
student_counts = {}   

for mentor_id, ratings in mentor_ratings.items():
    weights = get_weights_for_ratings(ratings)
    weighted_avg = np.average(ratings, weights=weights)
    feedback_scores[mentor_id] = weighted_avg.round(3)
    student_counts[mentor_id] = len(ratings)


In [66]:
feedback_scores

{'M001': np.float64(4.0),
 'M002': np.float64(3.667),
 'M003': np.float64(3.333),
 'M004': np.float64(3.0),
 'M005': np.float64(3.333)}

In [67]:
student_counts

{'M001': 3, 'M002': 3, 'M003': 3, 'M004': 1, 'M005': 3}

Bayesian Average

In [68]:
all_scores   = list(feedback_scores.values())
global_mean  = sum(all_scores) / len(all_scores)

In [69]:
global_mean

np.float64(3.4665999999999997)

In [70]:
k = 3

In [71]:
bayesian_feedback = {}

for mentor_id in feedback_scores:
    mean_rating = feedback_scores[mentor_id]
    n           = student_counts[mentor_id]
    
    adjusted = (n * mean_rating + k * global_mean) / (n + k)
    bayesian_feedback[mentor_id] = adjusted.round(3)
    

In [72]:
bayesian_feedback

{'M001': np.float64(3.733),
 'M002': np.float64(3.567),
 'M003': np.float64(3.4),
 'M004': np.float64(3.35),
 'M005': np.float64(3.4)}

In [73]:
feedback_series = pd.Series(bayesian_feedback) 

In [74]:
feedback_series

M001    3.733
M002    3.567
M003    3.400
M004    3.350
M005    3.400
dtype: float64

In [75]:
feedback_score  = (feedback_series - 1) / 4   

In [76]:
feedback_score

M001    0.68325
M002    0.64175
M003    0.60000
M004    0.58750
M005    0.60000
dtype: float64

In [77]:
feedback_score.name = "F"   

In [78]:
feedback_score

M001    0.68325
M002    0.64175
M003    0.60000
M004    0.58750
M005    0.60000
Name: F, dtype: float64

##### Combine All Scores into Final Score M

My Final formula
M = (0.35 × P) + (0.20 × R) + (0.25 × E) + (0.20 × F)

In [79]:
all_scores_table = pd.concat(
    [progress_score, responsiveness_score, engagement_score, feedback_score],
    axis=1 
)

In [80]:
all_scores_table

,P,R,E,F
M001,0.690000,0.8052,0.21,0.68325
M002,0.623333,0.7286,0.57,0.64175
M003,0.580000,0.4419,0.09,0.60000
M004,0.400000,0.6065,0.00,0.58750
M005,0.890000,0.8899,1.00,0.60000


In [81]:
w1 = 0.35
w2 = 0.20
w3 = 0.25
w4 = 0.20

P = 0.35 - This is most important because it shows if students actually learned something and completed their work.

E = 0.25 - This is second because meetings and reviews help us understand how much effort was put in.

R = 0.20 - This is also important because if mentors are slow, students can get frustrated.

F = 0.20 - This has a lower weight because feedback is based on opinion and can be different for everyone.

In [82]:
total_weight = w1 + w2 + w3 + w4

In [83]:
total_weight

1.0

In [84]:
all_scores_table["M"] = (
    w1 * all_scores_table["P"] +
    w2 * all_scores_table["R"] +
    w3 * all_scores_table["E"] +
    w4 * all_scores_table["F"]
)


In [85]:
all_scores_table

,P,R,E,F,M
M001,0.690000,0.8052,0.21,0.68325,0.591690
M002,0.623333,0.7286,0.57,0.64175,0.634737
M003,0.580000,0.4419,0.09,0.60000,0.433880
M004,0.400000,0.6065,0.00,0.58750,0.378800
M005,0.890000,0.8899,1.00,0.60000,0.859480


Score Evolution Over Time

### Exponential Smoothing Formula:

new_score = (1 - alpha) × old_score  +  alpha × current_score

again I am assuming alpha = 0.3

Current period - 30% weight

All previous history - 70% weight

In [86]:
def update_score(old_score, current_score, alpha=0.3):
    new_score = (1 - alpha) * old_score  +  alpha * current_score
    return new_score

In [87]:
old = 0.72
curr = 0.85
update_score(old, curr, alpha=0.3)

0.759

Activity Decay

new_score = old_score × (1 - decay_rate)

I am assuming decay_rate = 0.15 or we can say 15%

After 1 inactive period: score x 0.85

After 2 inactive periods: score x 0.85 x 0.85 = score x 0.72


I am assuming, if mentors not coming for 1 week but still they come and recover the reamining part.
so I want to give pnality for incative period but not so much. becoz may be there some reason to not continue, but that mentor can be good in overall score. so, too much panlity will be unfair I guess.

In [88]:
def apply_decay(score, decay_rate=0.15):
    return score * (1 - decay_rate)


In [89]:
score_before = 0.80

In [90]:
for period in range(1, 5):
    score_before = apply_decay(score_before)
    print(f"After {period} inactive period(s): {score_before:.4f}")

After 1 inactive period(s): 0.6800
After 2 inactive period(s): 0.5780
After 3 inactive period(s): 0.4913
After 4 inactive period(s): 0.4176


Final Ranked Output

In [91]:
all_scores_table

,P,R,E,F,M
M001,0.690000,0.8052,0.21,0.68325,0.591690
M002,0.623333,0.7286,0.57,0.64175,0.634737
M003,0.580000,0.4419,0.09,0.60000,0.433880
M004,0.400000,0.6065,0.00,0.58750,0.378800
M005,0.890000,0.8899,1.00,0.60000,0.859480


In [92]:
result = all_scores_table.reset_index()

In [93]:
result

,index,P,R,E,F,M
0,M001,0.690000,0.8052,0.21,0.68325,0.591690
1,M002,0.623333,0.7286,0.57,0.64175,0.634737
2,M003,0.580000,0.4419,0.09,0.60000,0.433880
3,M004,0.400000,0.6065,0.00,0.58750,0.378800
4,M005,0.890000,0.8899,1.00,0.60000,0.859480


In [94]:
result = result.rename(columns={"index": "MentorID"})

In [95]:
result

,MentorID,P,R,E,F,M
0,M001,0.690000,0.8052,0.21,0.68325,0.591690
1,M002,0.623333,0.7286,0.57,0.64175,0.634737
2,M003,0.580000,0.4419,0.09,0.60000,0.433880
3,M004,0.400000,0.6065,0.00,0.58750,0.378800
4,M005,0.890000,0.8899,1.00,0.60000,0.859480


In [96]:
result = pd.merge(
    result,
    mentor_table[["MentorID", "Name", "Domain"]],
    on="MentorID",
    how="left"
)

result

,MentorID,P,R,E,F,M,Name,Domain
0,M001,0.690000,0.8052,0.21,0.68325,0.591690,Alice Sharma,ML
1,M002,0.623333,0.7286,0.57,0.64175,0.634737,Bob Mehta,Web
2,M003,0.580000,0.4419,0.09,0.60000,0.433880,Carol Singh,CP
3,M004,0.400000,0.6065,0.00,0.58750,0.378800,Dev Patel,Systems
4,M005,0.890000,0.8899,1.00,0.60000,0.859480,Eva Nair,ML


In [97]:
result = result.sort_values("M", ascending=False)  

In [98]:
result

,MentorID,P,R,E,F,M,Name,Domain
4,M005,0.890000,0.8899,1.00,0.60000,0.859480,Eva Nair,ML
1,M002,0.623333,0.7286,0.57,0.64175,0.634737,Bob Mehta,Web
0,M001,0.690000,0.8052,0.21,0.68325,0.591690,Alice Sharma,ML
2,M003,0.580000,0.4419,0.09,0.60000,0.433880,Carol Singh,CP
3,M004,0.400000,0.6065,0.00,0.58750,0.378800,Dev Patel,Systems


In [99]:
result = result.reset_index(drop=True)

In [100]:
result

,MentorID,P,R,E,F,M,Name,Domain
0,M005,0.890000,0.8899,1.00,0.60000,0.859480,Eva Nair,ML
1,M002,0.623333,0.7286,0.57,0.64175,0.634737,Bob Mehta,Web
2,M001,0.690000,0.8052,0.21,0.68325,0.591690,Alice Sharma,ML
3,M003,0.580000,0.4419,0.09,0.60000,0.433880,Carol Singh,CP
4,M004,0.400000,0.6065,0.00,0.58750,0.378800,Dev Patel,Systems


In [101]:
result["Rank"] = result.index + 1

In [102]:
result

,MentorID,P,R,E,F,M,Name,Domain,Rank
0,M005,0.890000,0.8899,1.00,0.60000,0.859480,Eva Nair,ML,1
1,M002,0.623333,0.7286,0.57,0.64175,0.634737,Bob Mehta,Web,2
2,M001,0.690000,0.8052,0.21,0.68325,0.591690,Alice Sharma,ML,3
3,M003,0.580000,0.4419,0.09,0.60000,0.433880,Carol Singh,CP,4
4,M004,0.400000,0.6065,0.00,0.58750,0.378800,Dev Patel,Systems,5


In [103]:
final_table = result[["Rank", "MentorID", "Name", "Domain", "P", "R", "E", "F", "M"]]
final_table.columns = [
    "Rank", "MentorID", "Name", "Domain",
    "ProgressScore", "ResponsivenessScore", "EngagementScore",
    "FeedbackScore", "FinalScore_M"
]

In [104]:
final_table

,Rank,MentorID,Name,Domain,ProgressScore,ResponsivenessScore,EngagementScore,FeedbackScore,FinalScore_M
0,1,M005,Eva Nair,ML,0.890000,0.8899,1.00,0.60000,0.859480
1,2,M002,Bob Mehta,Web,0.623333,0.7286,0.57,0.64175,0.634737
2,3,M001,Alice Sharma,ML,0.690000,0.8052,0.21,0.68325,0.591690
3,4,M003,Carol Singh,CP,0.580000,0.4419,0.09,0.60000,0.433880
4,5,M004,Dev Patel,Systems,0.400000,0.6065,0.00,0.58750,0.378800


In [105]:
final_table.to_csv("mentor_scores.csv", index=False)